In [1]:
!pip install plotly scipy scikit-learn statsmodels

In [3]:
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
    OrdinalEncoder
)

from scipy.stats import chi2_contingency
from scipy.stats import pointbiserialr

import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

In [6]:
class PlottingMethods:

    """
    Modular plotting utilities
    """

    @staticmethod
    def bar_chart(df, column):

        if df is None or column not in df.columns:
            return None

        counts = df[column].value_counts()

        fig = px.bar(
            x=counts.index,
            y=counts.values,
            title=f"Bar Chart - {column}"
        )

        return fig.to_html()

    @staticmethod
    def pie_chart(df, column):

        if df is None or column not in df.columns:
            return None

        counts = df[column].value_counts()

        fig = px.pie(
            names=counts.index,
            values=counts.values,
            title=f"Pie Chart - {column}"
        )

        return fig.to_html()

    @staticmethod
    def histogram(df, column):

        if df is None or column not in df.columns:
            return None

        fig = px.histogram(df, x=column)

        return fig.to_html()


class DataInspector:

    def __init__(self):
        self.df = None

    def upload_data(self):

        uploaded = files.upload()

        filename = list(uploaded.keys())[0]

        self.df = pd.read_csv(
            filename,
            na_values=[
                '?',
                'n/a',
                'NULL',
                ' '
            ]
        )

        self.auto_type_correction()

        return self.df

    def auto_type_correction(self):

        for col in self.df.columns:

            converted = pd.to_numeric(
                self.df[col],
                errors='coerce'
            )

            if converted.notna().sum() > 0:
                self.df[col] = converted

    def data_summary(self):

        print("Rows:", self.df.shape[0])
        print("Columns:", self.df.shape[1])

        print("\nFirst 20 Rows\n")
        display(self.df.head(20))

        num_cols = self.df.select_dtypes(
            include=np.number
        ).columns

        cat_cols = self.df.select_dtypes(
            exclude=np.number
        ).columns

        print("\nNumeric Columns")
        print(list(num_cols))

        print("\nCategorical Columns")
        print(list(cat_cols))

    def handle_missing_values(
            self,
            strategy='mean',
            constant_value=0
    ):

        for col in self.df.columns:

            if self.df[col].isna().sum() == 0:
                continue

            if strategy == 'mean':

                if pd.api.types.is_numeric_dtype(
                        self.df[col]):
                    self.df[col] = self.df[col].fillna(
                        self.df[col].mean()
                    )

            elif strategy == 'median':

                if pd.api.types.is_numeric_dtype(
                        self.df[col]):
                    self.df[col] = self.df[col].fillna(
                        self.df[col].median()
                    )

            elif strategy == 'mode':

                self.df[col] = self.df[col].fillna(
                    self.df[col].mode()[0]
                )

            elif strategy == 'constant':

                self.df[col] = self.df[col].fillna(
                    constant_value
                )

    def remove_duplicates(self):

        before = len(self.df)

        self.df.drop_duplicates(
            inplace=True
        )

        after = len(self.df)

        print(
            f"Removed {before-after} duplicate rows"
        )

    def handle_outliers(
            self,
            column,
            delete=False
    ):

        q1 = self.df[column].quantile(0.25)

        q3 = self.df[column].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outliers = self.df[
            (self.df[column] < lower)
            |
            (self.df[column] > upper)
        ]

        print(
            "Outliers Found:",
            len(outliers)
        )

        if delete:

            self.df = self.df[
                (self.df[column] >= lower)
                &
                (self.df[column] <= upper)
            ]

    def delete_rows(self):

        rows = input(
            "Enter row indices:"
        )

        rows = [
            int(x)
            for x in rows.split(',')
        ]

        self.df.drop(
            rows,
            inplace=True
        )

    def delete_columns(self):

        cols = input(
            "Enter column names:"
        )

        cols = cols.split(',')

        self.df.drop(
            columns=cols,
            inplace=True
        )

    def extract_normalized_numeric_data(
            self,
            method='standard'
    ):

        num_df = self.df.select_dtypes(
            include=np.number
        )

        if method == 'minmax':
            scaler = MinMaxScaler()

        elif method == 'robust':
            scaler = RobustScaler()

        else:
            scaler = StandardScaler()

        scaled = scaler.fit_transform(
            num_df
        )

        return pd.DataFrame(
            scaled,
            columns=num_df.columns
        )

    def extract_normalized_categorical_data(
            self,
            method='onehot'
    ):

        cat_df = self.df.select_dtypes(
            exclude=np.number
        )

        if len(cat_df.columns) == 0:
            return pd.DataFrame()

        if method == 'onehot':

            encoder = OneHotEncoder(
                sparse_output=False
            )

            encoded = encoder.fit_transform(
                cat_df
            )

            return pd.DataFrame(
                encoded,
                columns=encoder.get_feature_names_out()
            )

        else:

            encoder = OrdinalEncoder()

            encoded = encoder.fit_transform(
                cat_df
            )

            encoded_df = pd.DataFrame(
                encoded,
                columns=cat_df.columns
            )

            if method == 'uniform':

                scaler = MinMaxScaler()

                encoded_df[:] = scaler.fit_transform(
                    encoded_df
                )

            return encoded_df

    def merge_normalized_data(
            self,
            num_method='standard',
            cat_method='onehot'
    ):

        num = self.extract_normalized_numeric_data(
            num_method
        )

        cat = self.extract_normalized_categorical_data(
            cat_method
        )

        return pd.concat(
            [num, cat],
            axis=1
        )

    def plot_numeric_distribution(
            self,
            column
    ):

        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=[
                "Violin/Box",
                "Scatter",
                "Histogram"
            ]
        )

        fig.add_trace(
            go.Violin(
                y=self.df[column]
            ),
            row=1,
            col=1
        )

        fig.add_trace(
            go.Scatter(
                y=self.df[column],
                mode='markers'
            ),
            row=1,
            col=2
        )

        fig.add_trace(
            go.Histogram(
                x=self.df[column]
            ),
            row=1,
            col=3
        )

        fig.show()

    def plot_relationship(
            self,
            col1,
            col2
    ):

        is_num1 = pd.api.types.is_numeric_dtype(
            self.df[col1]
        )

        is_num2 = pd.api.types.is_numeric_dtype(
            self.df[col2]
        )

        if is_num1 and is_num2:

            fig = px.scatter(
                self.df,
                x=col1,
                y=col2,
                trendline='ols'
            )

        elif is_num1 != is_num2:

            if is_num1:
                cat = col2
                num = col1
            else:
                cat = col1
                num = col2

            fig = px.box(
                self.df,
                x=cat,
                y=num,
                points='all'
            )

        else:

            temp = pd.crosstab(
                self.df[col1],
                self.df[col2]
            )

            fig = px.bar(
                temp
            )

        fig.show()

    def plot_categorical_frequency(
            self,
            column
    ):

        counts = self.df[column].value_counts()

        percent = (
            counts /
            counts.sum()
            * 100
        ).round(2)

        fig = px.bar(
            x=counts.index,
            y=counts.values,
            text=percent.astype(str)+"%"
        )

        fig.show()

    def plot_all_associations_heatmap(self):

        corr = self.df.corr(
            numeric_only=True
        )

        fig = px.imshow(
            corr,
            text_auto=True,
            title="Association Heatmap"
        )

        fig.show()

In [7]:
inspector = DataInspector()

# Upload Titanic CSV
inspector.upload_data()

# Summary
inspector.data_summary()

# Fill missing values
inspector.handle_missing_values(
    strategy='median'
)

# Remove duplicates
inspector.remove_duplicates()

# Example outlier handling
inspector.handle_outliers(
    'Age',
    delete=False
)

# Normalize
normalized_data = inspector.merge_normalized_data(
    num_method='standard',
    cat_method='onehot'
)

print(
    normalized_data.head()
)

# Visualization
inspector.plot_numeric_distribution(
    'Age'
)

inspector.plot_relationship(
    'Age',
    'Fare'
)

inspector.plot_relationship(
    'Sex',
    'Fare'
)

inspector.plot_categorical_frequency(
    'Sex'
)

inspector.plot_all_associations_heatmap()

Saving titanic.cvs.txt to titanic.cvs.txt
Rows: 891
Columns: 12

First 20 Rows



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C



Numeric Columns
['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']

Categorical Columns
['Name', 'Sex', 'Cabin', 'Embarked']
Removed 0 duplicate rows
Outliers Found: 66
   PassengerId  Survived    Pclass       Age     SibSp     Parch    Ticket  \
0    -1.730108 -0.789272  0.827377 -0.565736  0.432793 -0.473674 -0.044120   
1    -1.726220  1.266990 -1.566107  0.663861  0.432793 -0.473674 -0.044120   
2    -1.722332  1.266990  0.827377 -0.258337 -0.474545 -0.473674 -0.044120   
3    -1.718444  1.266990 -1.566107  0.433312  0.432793 -0.473674 -0.345494   
4    -1.714556 -0.789272  0.827377  0.433312 -0.474545 -0.473674  0.293977   

       Fare  Name_Abbing, Mr. Anthony  Name_Abbott, Mr. Rossmore Edward  ...  \
0 -0.502445                       0.0                               0.0  ...   
1  0.786845                       0.0                               0.0  ...   
2 -0.488854                       0.0                               0.0  ...   
3  0.420730